In [1]:
import pandas as pd

In [2]:
#loading data with needy attributes
df = pd.read_csv("Amazon Sale Report.csv", encoding="ISO-8859-1",usecols=['Order ID', 'Date', 'Status', 'Fulfilment', 'Sales Channel',
       'ship-service-level', 'Category', 'Size', 'Courier Status', 'Qty',
       'currency', 'Amount', 'ship-city', 'ship-state', 'ship-postal-code',
       'ship-country', 'B2B', 'fulfilled-by', 'New', 'PendingS'])


In [3]:
#checking missing values
df.isnull().sum()

Order ID                   0
Date                       0
Status                     0
Fulfilment                 0
Sales Channel              0
ship-service-level         0
Category                   0
Size                       0
Courier Status             0
Qty                        0
currency                7800
Amount                  7800
ship-city                 35
ship-state                35
ship-postal-code          35
ship-country              35
B2B                        0
fulfilled-by           89713
New                   128976
PendingS              128976
dtype: int64

In [4]:
pattern1=df[df['Amount'].isnull()]
pattern1.groupby(['Status','Courier Status'])['Order ID'].count().reset_index()


,Status,Courier Status,Order ID
0,Cancelled,Cancelled,5845
1,Cancelled,On the Way,1725
2,Cancelled,Unshipped,1
3,Pending,Cancelled,2
4,Shipped,Cancelled,93
5,Shipped,Unshipped,115
6,Shipped - Delivered to Buyer,On the Way,8
7,Shipped - Returned to Seller,On the Way,3
8,Shipping,Unshipped,8


<b>Conclusion : </b> <br>
we clearly found a pattern to fill the the cancelled amount value to zero either status or courier status is cancelled or unshipped, based on this pattern we can fill it with zero as the undelivered ordervalue is zero

In [5]:
def fill_amount(row):
    if (row['Status'] in ['Cancelled', 'Pending', 'Shipped - Returned to Seller']) or \
       (row['Courier Status'] in ['Cancelled', 'Unshipped']):
        return 0 if pd.isna(row['Amount']) else row['Amount']
    return row['Amount']

In [6]:
# Filling with zero based on concluded conditions
df['Amount'] = df.apply(fill_amount, axis=1)

In [7]:
#dropping three columns as they are hve no significance and filled with null values [fulfilled-by, New, PendingS] 
df = df.drop(columns=['fulfilled-by','New','PendingS'])

In [8]:
#as all the null valued currenceied records are within India we can fill those with INR
df['currency'] = df['currency'].fillna('INR')

In [9]:
#droping null record for ship-city, ship-state, ship-postal-code, ship-country as their count is negligible and they are categorical
df.dropna(subset=['ship-city', 'ship-state','ship-postal-code','ship-country'], inplace=True)

In [11]:
#checking is the data is properly filled or not and unwanted columns are dropped or not
df.isnull().sum()

Order ID              0
Date                  0
Status                0
Fulfilment            0
Sales Channel         0
ship-service-level    0
Category              0
Size                  0
Courier Status        0
Qty                   0
currency              0
Amount                8
ship-city             0
ship-state            0
ship-postal-code      0
ship-country          0
B2B                   0
dtype: int64

In [12]:
df.to_csv('clean_data.csv',index=False)